In [1]:
import sys
sys.path.append('../..')
import pandas as pd
from src.commands.knowledge_graph_construction.graph_cleaning.rule_based_predicate_cleaning import PredicatesNormalizer
from src.commands.knowledge_graph_construction.graph_cleaning.utils import (
    Predicate,
    parse_lhs,
    parse_rhs,
    parse_rule,
    Token,
    Tag,
    Placeholder,
    DELETE,
    KEEP,
    REPLACE,
    Rule,
    word_tokenize,
    pos_tag
)

In [2]:
from tqdm import tqdm
tqdm.pandas()

In [3]:
pos_tag(word_tokenize("can't"))

[('ca', 'MD'), ("n't", 'RB')]

In [3]:
df = pd.read_json('/home/fellah-a/Documents/Text2KG/data/knowledge_graphs/industry_of_future_v2/entities_normalization/data.jsonl', lines=True)

In [7]:
def process_predicate(s : str) -> str:

    s = s.replace('_', ' ')
    s = s.lower()
    s = s.strip()

    return s

In [10]:
predicates_df = df['predicate'].value_counts().reset_index()
predicates_df['predicate'] = predicates_df['predicate'].apply(process_predicate)
predicates_df['tokens'] = predicates_df['predicate'].apply(word_tokenize)
predicates_df['tags'] = predicates_df['tokens'].apply(pos_tag)
predicates_df['tag'] = predicates_df['tags'].apply(lambda x : ' '.join(tag for word, tag in x))
predicates_df.head()

,predicate,count,tokens,tags,tag
0,includes,2948,[includes],"[(includes, VBZ)]",VBZ
1,of,1699,[of],"[(of, IN)]",IN
2,based on,1461,"[based, on]","[(based, VBN), (on, IN)]",VBN IN
3,is,1409,[is],"[(is, VBZ)]",VBZ
4,has property,1269,"[has, property]","[(has, VBZ), (property, NN)]",VBZ NN


In [13]:
predicates_df[predicates_df['predicate'].str.startswith('have')][['tag', 'count']].groupby('tag').sum().reset_index()

,tag,count
0,VB,593
1,VB CD JJ IN NN TO,1
2,VB DT JJ,1
3,VB DT JJ NN IN,4
4,VB DT NN IN RB CD NNS CD,1
...,...,...
69,VBP VBN DT NN IN,1
70,VBP VBN IN,3
71,VBP VBN NNS IN,1
72,VBP VBN VBN,1


In [16]:
predicates_df[(predicates_df['predicate'].str.startswith('have')) & (predicates_df['tag'].str.startswith('VB NN'))]

,predicate,count,tokens,tags,tag
2324,have position in,7,"[have, position, in]","[(have, VB), (position, NN), (in, IN)]",VB NN IN
3633,have applications in,4,"[have, applications, in]","[(have, VB), (applications, NNS), (in, IN)]",VB NNS IN
4124,have origin,4,"[have, origin]","[(have, VB), (origin, NN)]",VB NN
4582,have properties,3,"[have, properties]","[(have, VB), (properties, NNS)]",VB NNS
4810,have application in,3,"[have, application, in]","[(have, VB), (application, NN), (in, IN)]",VB NN IN
5018,have safety barriers depending upon,3,"[have, safety, barriers, depending, upon]","[(have, VB), (safety, NN), (barriers, NNS), (d...",VB NN NNS VBG IN
5080,have improvements in,3,"[have, improvements, in]","[(have, VB), (improvements, NNS), (in, IN)]",VB NNS IN
5899,have length,3,"[have, length]","[(have, VB), (length, NN)]",VB NN
7221,have insufficient,2,"[have, insufficient]","[(have, VB), (insufficient, NN)]",VB NN
7940,have shape,2,"[have, shape]","[(have, VB), (shape, NN)]",VB NN


In [4]:
df.head()

,id,head,predicate,tail,sentence
0,542065800,construction sector,contributes_to,global greenhouse gas emission,"{'id': '5420658_0', 'sentence': 'The construct..."
1,542065800,construction sector,contributes_to,global greenhouse gas emission,"{'id': '5420658_0', 'sentence': 'The construct..."
2,542065800,construction sector,contributes_to,global greenhouse gas emission,"{'id': '5420658_0', 'sentence': 'The construct..."
3,542065800,construction sector,contributes_to,global greenhouse gas emission,"{'id': '5420658_0', 'sentence': 'The construct..."
4,542065800,construction sector,contributes_to,global greenhouse gas emission,"{'id': '5420658_0', 'sentence': 'The construct..."


In [20]:
records = df.to_dict(orient='records')

In [6]:
normalizer = PredicatesNormalizer(None, None)

In [ ]:
44561

In [7]:
mapped = []

for record in tqdm(records):
    try:
        mapped.append(normalizer.transform(record))
    except Exception as e:
        print(e)
        print(record['predicate'])
        break

100%|██████████| 158815/158815 [08:07<00:00, 325.85it/s]


In [8]:
filtered_mapped = list(filter(None, mapped))

In [11]:
predicates = df['predicate']

In [15]:
predicates.value_counts(normalize=True).cumsum().lt(0.8).sum()

np.int64(1327)

In [18]:
predicates[predicates.str.contains('challenge')].value_counts()

predicate
challenge for               97
challenge                   81
challenge in                40
have challenge              28
challenge of                15
                            ..
constitute challenge for     1
be challenge when            1
challenge because            1
challenge for power          1
be challenge issue in        1
Name: count, Length: 106, dtype: int64

In [13]:
len(df)

122228

In [1]:
records

NameError: name 'records' is not defined

In [14]:
predicates.value_counts()

predicate
include               3470
be                    2585
base on               1666
cause                 1477
have property         1429
                      ... 
dilute in                1
permit decrease of       1
objective to             1
offer path to            1
equalize with            1
Name: count, Length: 14256, dtype: int64

In [14]:
record

{'id': 4696837140,
 'head': 'result',
 'predicate': '',
 'tail': 'result',
 'sentence': {'id': '4696837_14',
  'sentence': 'RESULTS:.',
  'document': {'id': '4696837',
   'title': 'Table tennis performance indicators relationship under fatigue condition',
   'abstract': 'INTRODUCTION:Table tennis player performance has already been defined in the literature by several aspects such as tactics,equipment, and technique. Ball quality of stroke could be one of them, especially ball speed, rotation, or evenprecision. Few studies have investigated the relationship between those indicators on different strokeconditions and under neuromuscular or mental fatigue which has been proven to be an influential factor instroke performance. This study aims to observe the effect of fatigue on the relationship between theseperformance indicators and their evolution in fatigue.METHODS:Nine young table tennis players men (age :15 ±1.5y; French ranking points: 2103±344) from a French table tennisnational cen

In [4]:
normalizer.normalize_predicate("can be used to predict").format(head="model", tail="risk").split(' | ')

['model', 'predict', 'risk']

In [10]:
normalizer.transform({
    "id" : "5ra",
    "sentence" : "zbal",
    "head" : "risk",
    "predicate" : "predicted by using",
    "tail" : "model"
})

{'id': '5ra',
 'sentence': 'zbal',
 'head': 'model',
 'predicate': 'predict',
 'tail': 'risk'}